# Data Preprocessing and Citation Network Analysis with Author Influence
Fariha Adil, 2375026

## About
The goal of this analysis is to preprocess the DBLP dataset and build a citation network to identify influential papers and authors. Graph-based metrics like PageRank and in-degree centrality are used to measure influence, which is then tracked over time to reveal how impact evolves and which papers represent true breakthroughs in computer science research.

## Tasks
- Preprocess the raw DBLP dataset: handle missing data, filter sparse venues, normalize author names, and document dataset statistics
- Build a directed citation network where nodes are papers and edges represent citation relationships
- Use graph metrics (in-degree centrality, PageRank, betweenness centrality) to identify influential papers and authors
- Combine influence scores with temporal data to track how author and paper impact evolves over time
- Identify breakthrough papers that shifted research directions using citation velocity and PageRank

## Motivations
- Understand which papers and authors have truly shaped computer science research beyond simple citation counts
- Identify emerging researchers and paradigm-shifting work that may go unnoticed by raw citation metrics alone
- Track how influence evolves: does an author's impact grow over time or fade? Do seminal papers maintain relevance?
- Discover research momentum: which areas are gaining traction vs. declining

## Challenges
- Computational complexity: building and analyzing networks with millions of nodes and edges requires efficient graph algorithms and working with filtered subsets
- Incomplete citation data: not all papers cite all prior relevant work; citations reflect only the subset recorded in DBLP
- Temporal dynamics: influence changes over time; time windows must be carefully defined and papers with delayed impact must be accounted for
- Bias in metrics: PageRank and centrality favor well-established papers; must balance with novelty metrics like citation velocity
- Author disambiguation: the same name can refer to different authors, making author-level influence aggregation imprecise

In [4]:
"""Import Dependencies"""
import json
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

In [5]:
"""Globals/Constants"""
DATA_PATH = './dblp_ref/'

START_YEAR = 2000
END_YEAR = 2017

MIN_VENUE_COUNT = 10

SAMPLE_SIZE = 200000
RANDOM_STATE = 42

In [6]:
"""Helper/Utility Functions"""
def read_json_lines(file_path):
    """Reads the JSON file at the given path and yields each line (record)."""
    with open(file_path) as json_file:
        for line in json_file:
            yield json.loads(line)

def normalize_author(name):
    return name.strip().lower() if isinstance(name, str) else None

In [7]:
"""Load Data"""
frames = []
for file_index in range(4):
    file_name = f'{DATA_PATH}dblp-ref-{file_index}.json'
    frames.append(pd.DataFrame(read_json_lines(file_name)))

papers = pd.concat(frames, ignore_index = True)
papers.head()

,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[Makoto Satoh, Ryo Muramatsu, Mizue Kayama, Ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[Gareth Beale, Graeme Earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[Altaf Hossain, Faisal Zaman, Mohammed Nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[Jea-Bum Park, Byungmok Kim, Jian Shen, Sun-Yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
4,NaN,"[Giovanna Guerrini, Isabella Merlo]",2,NaN,Reasonig about Set-Oriented Methods in Object ...,,1998,0040b022-1472-4f70-a753-74832df65266


In [ ]:
"""Preprocessing: Finding Missing Data in Dataset"""
total = len(papers)

missing_venue = papers['venue'].isna().sum()
missing_references = papers['references'].isna().sum()
missing_abstract = papers['abstract'].isna().sum()
missing_authors = papers['authors'].isna().sum()

print(f"Total papers: {total:,}")
print(f"Missing venue: {missing_venue:,} ({missing_venue/total*100:.0f}%)")
print(f"Missing references: {missing_references:,} ({missing_references/total*100:.2f}%)")
print(f"Missing abstract: {missing_abstract:,} ({missing_abstract/total*100:.2f}%)")
print(f"Missing authors: {missing_authors:,} ({missing_authors/total*100:.4f}%)")

citation_counts = papers['n_citation']
print(f"\nCitation Count Statistics:")
print(f"Mean: {citation_counts.mean():.2f}")
print(f"Median: {citation_counts.median():.0f}")
print(f"Max: {citation_counts.max():,}")
print(f"Zero citations: {(citation_counts == 0).sum():,} ({(citation_counts == 0).sum()/total*100:.2f}%)")

Total papers: 3,079,007
Missing venue: 0 (0%)
Missing references: 362,865 (11.79%)
Missing abstract: 530,475 (17.23%)
Missing authors: 4 (0.0001%)

Citation Count Statistics:
Mean: 35.22
Median: 11
Max: 73,362
Zero citations: 718,250 (23.33%)


In [45]:
"""Preprocessing: Filtering and Cleaning"""
cleaned = papers.copy()

cleaned = cleaned[(cleaned['year'] >= START_YEAR) & (cleaned['year'] <= END_YEAR)]
print(f"After filtering by years ({START_YEAR} - {END_YEAR}): {len(cleaned):,}")

cleaned = cleaned.dropna(subset = ['title', 'authors', 'id'])
print(f"After dropping incomplete rows: {len(cleaned):,}")

cleaned['authors'] = cleaned['authors'].apply(lambda author_list: [normalize_author(a) for a in author_list] if isinstance(author_list, list) else [])

cleaned['venue'] = cleaned['venue'].fillna('')
venue_counts = cleaned['venue'].value_counts()
valid_venues = venue_counts[venue_counts >= MIN_VENUE_COUNT].index
cleaned['venue'] = cleaned['venue'].apply(lambda v: v if v in valid_venues else 'Other')
print(f"Venues after filtering: {cleaned['venue'].nunique():,}")

cleaned['references'] = cleaned['references'].apply(lambda refs: refs if isinstance(refs, list) else [])

print(f"\nFinal cleaned dataset: {len(cleaned):,} papers")
cleaned.head()

After filtering by years (2000 - 2017): 2,676,427
After dropping incomplete rows: 2,676,423
Venues after filtering: 3,043

Final cleaned dataset: 2,676,423 papers


,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[makoto satoh, ryo muramatsu, mizue kayama, ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[gareth beale, graeme earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[altaf hossain, faisal zaman, mohammed nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[jea-bum park, byungmok kim, jian shen, sun-yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
5,NaN,"[rafael álvarez, leandro tortosa, josé-francis...",0,[],COMPARING GNG3D AND QUADRIC ERROR METRICS METH...,international conference on computer graphics ...,2009,005ce28f-ed77-4e97-afdc-a296137186a1


In [47]:
"""Sample for Graph Analysis"""
if SAMPLE_SIZE is not None and len(cleaned) > SAMPLE_SIZE:
    sample = cleaned.sample(n = SAMPLE_SIZE, random_state = RANDOM_STATE).copy()
else:
    sample = cleaned.copy()

print(f"Working sample size: {len(sample):,}")

Working sample size: 200,000


In [49]:
"""Build Citation Network"""
paper_ids = set(sample['id'])

citation_graph = nx.DiGraph()
citation_graph.add_nodes_from(sample['id'])

edge_count = 0
for paper in sample.itertuples():
    for ref_id in paper.references:
        if ref_id in paper_ids:
            citation_graph.add_edge(paper.id, ref_id)
            edge_count += 1

print(f"Nodes: {citation_graph.number_of_nodes():,}")
print(f"Edges: {citation_graph.number_of_edges():,}")
print(f"Is DAG: {nx.is_directed_acyclic_graph(citation_graph)}")

Nodes: 200,000
Edges: 102,204
Is DAG: False
